# 05 — Kepler.gl Visualization

## 1. Introduction

Produces polished, standalone kepler.gl maps from the scored access data,
purely for **visual storytelling**. This is separate from the interactive
Streamlit dashboard (`dashboard/app.py`), which is the project's
primary interactive analysis tool.
 Kepler.gl is used here specifically for
its strong default cartography and easy exportable HTML which is well suited for
embedding in the ArcGIS StoryMap, the GitHub README, or the written report
as a supplementary visual.

### Maps produced

1. **Combined access-deficit map** — both LGAs, walking mode, the primary
   "headline" visual for the submission.
2. **Mode comparison map** — walk vs. okada vs. drive, toggleable layers,
   for one LGA — the visual expression of the project's multi-modal
   accessibility angle.
3. **Completeness flag map** — where OSM data itself may be incomplete,
   tying back to Notebook 02's analysis.

### Why an ordinal color scale, not a continuous gradient

The access-deficit score is a discrete category (0, 1, or 2), not a
continuous measurement — so the kepler configs used here (`colorScale:
"ordinal"` with an explicit 3-color domain) map each score directly to a
fixed color. A continuous/quantized color scale would instead bin
whatever range of values happens to be *present* in the data, which can
silently collapse to fewer visible colors if one score value is rare
(this was an actual bug caught and fixed during this project's
development — see `kepler_config_access_deficit.json`'s comments).

### Expected outputs

Three standalone HTML files, each fully self-contained (data + viewer
bundled), viewable in any browser without Python installed (see Section 7,
Export).

### Prerequisites

- Notebooks 01–04 have been run for both study areas.
- `pip install keplergl` (installed automatically by the Colab setup cell
  below).


## 2. Imports

`keplergl` is the one dependency specific to this notebook — everything
else (`geopandas`, `pandas`, `json`) is already familiar from earlier
notebooks.

### 2.1 Environment setup

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
DASHBOARD_DRIVE_FOLDER = "Akure Access Dashboard"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DASHBOARD_DIR = f"/content/drive/MyDrive/{DASHBOARD_DRIVE_FOLDER}"
    if not os.path.exists(DASHBOARD_DIR):
        raise FileNotFoundError(f"Expected '{DASHBOARD_DIR}' in your Google Drive.")

    %cd {DASHBOARD_DIR}/notebooks
    !pip install geopandas shapely fiona pandas keplergl --quiet
    print(f"Running in Colab. Working directory: {os.getcwd()}")
else:
    print("Running locally.")


### 2.2 Package imports

In [ ]:
import json
import geopandas as gpd
import pandas as pd
from keplergl import KeplerGl

## 3. Configuration

This notebook's real "configuration" is the three `kepler_config_*.json`
files in this folder, which capture exact layer styling, color ramps, and
map center/zoom (see Section 6 for how these are maintained).

## 4. Data Loading

### 4.1 Load combined dataset

Loads the combined scored dataset produced by Notebook 04, **reprojected
to EPSG:4326 (WGS84)** — kepler.gl, like most web-mapping libraries,
expects plain latitude/longitude coordinates rather than the projected
metric CRS (EPSG:32631) used throughout the rest of the analysis pipeline.
This is a one-way trip for visualization purposes only: no further
analysis happens in this notebook, so reprojecting away from the metric
CRS here doesn't affect anything upstream.

In [ ]:
import json
import geopandas as gpd
import pandas as pd
from keplergl import KeplerGl

combined = gpd.read_file("../data/processed/combined_access_scored.geojson")
combined_wgs84 = combined.to_crs("EPSG:4326")

settled = combined_wgs84[combined_wgs84["building_count"] > 0].copy()
print(f"Loaded {len(settled)} settled grid cells across both LGAs.")
settled.head()


## 5. Processing

### 5.1 Map 1 — Combined access-deficit map (both LGAs, walking mode)

The primary visual for the submission: every settled cell in both study
areas, colored by walking access-deficit score (0 = adequately served for
both health and education, 2 = underserved for both).

In [ ]:
with open("kepler_config_access_deficit.json") as f:
    access_config = json.load(f)

map_1 = KeplerGl(height=600, data={"akure_access": settled}, config=access_config)
map_1


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

### 5.2 Map 2 — Mode comparison map (walk vs. okada vs. drive)

Three toggleable layers for Akure North, letting a viewer flip between
transport-mode assumptions on the same map. This is the visual expression
of the project's multi-modal accessibility angle — walking-only analysis
tends to overstate the severity of the accessibility problem in a context
where okada is the practical norm.

In [ ]:
mode_cols = [
    "cell_id", "building_count", "geometry",
    "health_time_min_walk", "health_time_min_okada", "health_time_min_drive",
    "walk_access_deficit_score", "okada_access_deficit_score", "drive_access_deficit_score",
]

north_modes = gpd.read_file("../data/processed/akure_north/grid_access_scored.geojson")
north_modes_wgs84 = north_modes.to_crs("EPSG:4326")
available_cols = [c for c in mode_cols if c in north_modes_wgs84.columns]
mode_compare = north_modes_wgs84[available_cols]

print(f"Loaded {len(mode_compare)} cells for Akure North mode comparison.")
mode_compare.head()


In [ ]:
with open("kepler_config_mode_comparison.json") as f:
    mode_config = json.load(f)

map_2 = KeplerGl(height=600, data={"akure_north_modes": mode_compare}, config=mode_config)
map_2


### 5.3 Map 3 — OSM completeness flag map

Shows where OSM's own facility data may be incomplete (settled cells with
no nearby OSM health/education tag), tying back to Notebook 02. Presenting
this as its own map — rather than folding it silently into the access map —
keeps the "is this a service gap or a data gap?" distinction visible and
honest in the submission's visual narrative.

In [ ]:
completeness_cols = ["cell_id", "building_count", "geometry", "health_completeness_flag", "education_completeness_flag"]
available_completeness_cols = [c for c in completeness_cols if c in settled.columns]

if len(available_completeness_cols) > 2:
    completeness_data = settled[available_completeness_cols]

    # Reuse the same free, token-free CARTO basemap style as Maps 1 and
    # 2 (see Section 6.0 below), rather than leaving this map on
    # kepler.gl's internal default -- which is Mapbox-hosted and would
    # otherwise be visually inconsistent with the other two maps once
    # the embedded token is stripped from all three exports.
    with open("kepler_config_access_deficit.json") as f:
        _base_config = json.load(f)
    map_3_config = {"version": "v1", "config": {"mapStyle": _base_config["config"]["mapStyle"]}}

    map_3 = KeplerGl(height=600, data={"akure_completeness": completeness_data}, config=map_3_config)
    map_3
else:
    print("Completeness flag columns not found in the combined dataset — "
          "ensure Notebook 02 was run before Notebook 04 combined the data.")

## 6. Visualization: Exporting the Maps

Each map built above is exported to a standalone HTML file. These are
covered here as a distinct step from Section 5's map *construction*,
since exporting is a deliberate, separate action (you could keep iterating
on a map's styling live in the widget above before deciding it's ready to
export).

### 6.0 A note before saving: stripping the bundled Mapbox token

The installed `keplergl` package bakes a real (if publicly-scoped)
Mapbox access token directly into every `save_to_html()` export --
confirmed to happen even when, as here, the configured basemap style
is a free CARTO style rather than a Mapbox-hosted one (the token
appears to be embedded in keplergl's bundled JS regardless of which
style is actually selected, likely for an unrelated internal feature
such as the in-app style switcher). This is not something this
notebook's config controls.

GitHub's push protection will correctly flag this token if these files
are committed as-is (this happened once already during this project's
development). The helper below strips it from each exported file
immediately after saving. Since all three maps use the free CARTO
Positron basemap (see Section 5.1's config, and Map 3's config above)
rather than a Mapbox-hosted style, stripping this token has **no
visual effect at all** -- it only removes an unused, embedded
credential that the displayed basemap was never actually relying on.

In [ ]:
import re

# See the markdown note above for why this is necessary. Matches the
# same token pattern used in the companion lga-osm-extractor repo's
# visualize.py -- if you're comparing the two, this is deliberately
# kept consistent rather than reinvented per-repo.
_MAPBOX_TOKEN_PATTERN = re.compile(r"pk\.eyJ[A-Za-z0-9_\-]+\.[A-Za-z0-9_\-]+")


def strip_mapbox_token(html_path):
    """Remove any bundled Mapbox access token from an exported kepler.gl
    HTML file, in place. Returns True if a token was found and removed."""
    with open(html_path, "r", encoding="utf-8") as f:
        content = f.read()
    stripped, n_replaced = _MAPBOX_TOKEN_PATTERN.subn("", content)
    if n_replaced > 0:
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(stripped)
    return n_replaced > 0

In [ ]:
os.makedirs("../visuals", exist_ok=True)
map_1.save_to_html(file_name="../visuals/akure_access_deficit_map.html")
strip_mapbox_token("../visuals/akure_access_deficit_map.html")
print("Saved: visuals/akure_access_deficit_map.html")

In [ ]:
map_2.save_to_html(file_name="../visuals/akure_north_mode_comparison_map.html")
strip_mapbox_token("../visuals/akure_north_mode_comparison_map.html")
print("Saved: visuals/akure_north_mode_comparison_map.html")

In [ ]:
if len(available_completeness_cols) > 2:
    map_3.save_to_html(file_name="../visuals/akure_completeness_map.html")
    strip_mapbox_token("../visuals/akure_completeness_map.html")
    print("Saved: visuals/akure_completeness_map.html")

In [ ]:
import re
for f in ["akure_access_deficit_map.html", "akure_north_mode_comparison_map.html", "akure_completeness_map.html"]:
    path = f"../visuals/{f}"
    content = open(path).read()
    has_token = bool(re.search(r"pk\.eyJ1IjoidWNmLW1hcGJveCI", content))
    print(f"{f}: token present = {has_token}")

### 6.1 Reproducible styling

The `kepler_config_*.json` files in this folder capture the exact layer
styling, color ramps, and map center/zoom used above, so re-running this
notebook produces visually consistent maps rather than relying on manual
kepler.gl UI adjustments each time.

To adjust styling: tweak layers/colors interactively in the kepler.gl
widget above, then run the cell below to print the updated config and save
it back into the corresponding JSON file.

In [ ]:
# Example: after adjusting map_1 interactively above, save the updated config:
#
# with open("kepler_config_access_deficit.json", "w") as f:
#     json.dump(map_1.config, f, indent=2)
#
# print("Config updated.")


## 7. Export

**Outputs produced:**

```
visuals/akure_access_deficit_map.html          (Map 1 — combined, both LGAs)
visuals/akure_north_mode_comparison_map.html   (Map 2 — walk/okada/drive toggle)
visuals/akure_completeness_map.html             (Map 3 — OSM data-gap flags)
```

Each HTML file is fully standalone (data + kepler.gl viewer bundled),
with the bundled Mapbox token stripped (see Section 6.0), and can be:
- embedded directly in the ArcGIS StoryMap as an embedded web page,
- linked from the GitHub README,
- opened locally by anyone reviewing the submission without needing
  Python installed (basemap tiles will show blank rather than street
  imagery, since the token was intentionally removed -- all data
  layers still render and are fully interactive).

## 8. Summary

**What this notebook accomplished:** produced three polished, standalone,
shareable map visuals from the scored accessibility data, using a
discrete/ordinal color scheme matched to the data's actual categorical
structure.

**This completes the analysis pipeline.** The full notebook sequence
(01 → 05) takes raw OSM data through extraction, completeness assessment,
multi-modal accessibility scoring, cross-area/cross-mode summarization, and
finally polished visual output — matching the methodology documented in
the project's written methodology document and README.